# Step 8 - Collect and check the human annotations

Run this notebook twice:
1. **After the pilot** (`MODE = "pilot"`): shows how much the annotators agree per question and which wording is confusing, so the rubric can be fixed before the main round.
2. **After the main round** (`MODE = "main"`): validates every returned file, merges them into `processed/annotations/annotations_raw.csv` (one row per annotator x item) and reports completeness and agreement.

Put the CSV files the annotators send back (`ratings_pilot_A1.csv`, `ratings_main_A2.csv`, ...) into `processed/annotations/incoming/`.

Agreement here is a quick check only (percent agreement, Cohen's kappa for yes/no questions, "within 1 point" for 1-5 scales). The full analysis, including Krippendorff's alpha and judge-vs-human agreement, is Step 10.

In [7]:
import sys, json, itertools
from pathlib import Path
import numpy as np, pandas as pd


def find_project_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "src" / "config.py").exists():
            return p
    raise FileNotFoundError("Open the notebook from inside the FoodJudgeBench folder.")


ROOT = find_project_root(); sys.path.insert(0, str(ROOT / "src"))
from config import PROC_DIR

MODE = "pilot"            # "pilot" or "main"
BANK = PROC_DIR / "case_bank"
ANN = PROC_DIR / "annotations"; (ANN / "incoming").mkdir(parents=True, exist_ok=True)
spec = json.load(open(BANK / "rubric.json", encoding="utf-8"))
RUBRIC, COLS = spec["rubric"], spec["columns"]
YN = [r["id"] for r in RUBRIC if r["type"].startswith("yn")]
SCALE = [r["id"] for r in RUBRIC if r["type"].startswith("scale")]
key = pd.read_csv(BANK / "answer_key.csv").set_index("case_id")

#### 1. Load and validate the returned files

In [8]:
files = sorted((ANN / "incoming").glob(f"ratings_{MODE}_*.csv"))
assert files, f"no ratings_{MODE}_*.csv files in {ANN/'incoming'}"
frames, issues = [], []
for f in files:
    d = pd.read_csv(f, dtype=str, keep_default_na=False)
    miss_cols = set(COLS) - set(d.columns)
    if miss_cols:
        issues.append(f"{f.name}: missing columns {sorted(miss_cols)}"); continue
    d["_file"] = f.name; frames.append(d)
raw = pd.concat(frames, ignore_index=True)
if MODE == "main":
    assigned = {a: set(pd.read_csv(BANK / f"assignment_{a}.csv")["case_id"]) for a in raw["annotator_id"].unique()}
else:
    assigned = {a: set(pd.read_csv(BANK / "pilot_case_ids.csv")["case_id"]) | set() for a in raw["annotator_id"].unique()}
for a, g in raw.groupby("annotator_id"):
    dup = g["case_id"].duplicated().sum()
    extra = set(g["case_id"]) - assigned[a]
    todo = assigned[a] - set(g["case_id"])
    if dup: issues.append(f"{a}: {dup} duplicate rows")
    if extra: issues.append(f"{a}: {len(extra)} items that were not assigned to this annotator")
    print(f"{a}: {len(g)} rated, {len(todo)} still missing")
cases = raw[raw["item_type"].isin(["case", "diagnostic"])]
bad = []
for r in RUBRIC:
    ok = {"yn": {"yes", "no"}, "yn_na": {"yes", "no", "na"}, "scale": set("12345"), "scale_na": set("12345") | {"na"}}[r["type"]]
    b = cases[~cases[r["id"]].isin(ok)]
    if len(b): bad.append((r["id"], len(b)))
if bad: issues.append(f"invalid / empty answers: {bad}")
print("\nISSUES:" if issues else "\nNo validation issues."); [print(" -", i) for i in issues]

A1: 10 rated, 0 still missing
A2: 10 rated, 0 still missing
A3: 10 rated, 0 still missing

No validation issues.


[]

#### 2. Save the merged raw data

In [9]:
raw.drop(columns="_file").to_csv(ANN / f"annotations_{MODE}_raw.csv", index=False)
print("saved", ANN / f"annotations_{MODE}_raw.csv", raw.shape)
print("raters per item:", raw.groupby("case_id")["annotator_id"].nunique().value_counts().to_dict(), "(goal: >= 2, planned 3)")
print("median seconds per item:", pd.to_numeric(raw["seconds"], errors="coerce").median())

saved C:\HP Data\University-Masters\Semester_3\P1_Natural Language Processing\Lab\Project\FoodJudgeBench\processed\annotations\annotations_pilot_raw.csv (30, 20)
raters per item: {3: 10} (goal: >= 2, planned 3)
median seconds per item: 393.5


#### 3. Quick agreement per question (use after the pilot to find confusing wording)

In [10]:
def cohen_kappa(a, b):
    a, b = list(a), list(b); cats = sorted(set(a) | set(b)); n = len(a)
    if n == 0: return np.nan
    po = sum(x == y for x, y in zip(a, b)) / n
    pe = sum((a.count(c) / n) * (b.count(c) / n) for c in cats)
    return np.nan if pe == 1 else (po - pe) / (1 - pe)


rows = []
for q in YN + SCALE:
    pairs = []
    for cid, g in cases.groupby("case_id"):
        vals = g[q].tolist()
        pairs += list(itertools.combinations(vals, 2))
    if not pairs:
        continue
    agree = np.mean([x == y for x, y in pairs])
    row = {"question": q, "pairs": len(pairs), "exact_agreement": round(agree, 2)}
    if q in YN:
        row["kappa"] = round(cohen_kappa([p[0] for p in pairs], [p[1] for p in pairs]), 2)
    else:
        num = [(int(x), int(y)) for x, y in pairs if x != "na" and y != "na"]
        row["within_1_point"] = round(np.mean([abs(x - y) <= 1 for x, y in num]), 2) if num else np.nan
        row["na_disagreement"] = round(np.mean([(x == "na") != (y == "na") for x, y in pairs]), 2)
    rows.append(row)
agr = pd.DataFrame(rows).set_index("question"); print(agr.to_string())
print("\nRule of thumb: exact agreement < 0.7 (yes/no) or within-1 < 0.8 (scales) means the wording of that question needs work.")

                          pairs  exact_agreement  kappa  within_1_point  na_disagreement
question                                                                                
ignored_constraint           30             0.57   0.10             NaN              NaN
factual_error                30             0.73   0.37             NaN              NaN
invented_personalization     30             0.73   0.47             NaN              NaN
unsupported_claim            30             0.67   0.26             NaN              NaN
overstated_certainty         30             0.57   0.33             NaN              NaN
clarity                      30             0.33    NaN            0.87             0.00
profile_fit                  30             0.23    NaN            0.80             0.00
goal_alignment               30             0.80    NaN            1.00             0.13
tradeoff                     30             0.30    NaN            0.79             0.20
overall              

#### 4. Do annotators catch the injected errors? (diagnostic items)

In [11]:
if MODE and (cases["item_type"] == "diagnostic").any():
    FLAG = {"invented_preference": "invented_personalization", "wrong_nutrition": "factual_error",
            "ignored_time_limit": "ignored_constraint", "ignored_dietary_constraint": "ignored_constraint"}
    dg = cases[cases["item_type"] == "diagnostic"].join(key[["injected_error"]], on="case_id")
    dg["caught"] = [r[FLAG[r["injected_error"]]] == "yes" for _, r in dg.iterrows()]
    print(dg.groupby("injected_error")["caught"].agg(["mean", "count"]).round(2).rename(columns={"mean": "share_of_ratings_that_flag_it"}))
    print("(low values mean the question wording or the example error is unclear; fix before the main round)")

                     share_of_ratings_that_flag_it  count
injected_error                                           
invented_preference                           0.83      6
(low values mean the question wording or the example error is unclear; fix before the main round)


#### 5. Pairwise items

In [12]:
pr = raw[raw["item_type"] == "pair"]
if len(pr):
    pk = pd.read_csv(BANK / "answer_key.csv").set_index("case_id")
    pr = pr.join(pk[["A", "B"]], on="case_id")
    pr["winner"] = np.where(pr["pair_choice"] == "A", pr["A"], np.where(pr["pair_choice"] == "B", pr["B"], "tie"))
    print(pr["winner"].value_counts().to_string())
    print("agreement among annotators on the winner:", round(np.mean([len(set(g["pair_choice"])) == 1 for _, g in pr.groupby("case_id")]), 2))